# META-CXR — report-generation NLP metrics

Generates aggregate-only test metrics; it never writes MIMIC references or generated reports to disk.

In [ ]:
DATASET_SLUG = 'REPLACE_WITH_PRIVATE_MIMIC_DATASET'
CHECKPOINT_GCS_BUCKET = 'REPLACE_WITH_PRIVATE_CHECKPOINT_BUCKET'
REPO_COMMIT = 'REPLACE_WITH_EXACT_40_CHARACTER_SHA'
BATCH_SIZE = 2
NUM_WORKERS = 4
NUM_BEAMS = 1
MAX_LENGTH = 128
MIN_LENGTH = 8


In [ ]:
import os, pathlib, subprocess, sys
if not DATASET_SLUG or DATASET_SLUG.startswith('REPLACE_') or not CHECKPOINT_GCS_BUCKET or CHECKPOINT_GCS_BUCKET.startswith('REPLACE_'):
    raise ValueError('Set DATASET_SLUG and CHECKPOINT_GCS_BUCKET')
if len(REPO_COMMIT) != 40 or any(c not in '0123456789abcdef' for c in REPO_COMMIT.lower()):
    raise ValueError('REPO_COMMIT must be an exact 40-character SHA')
repo_dir = pathlib.Path('/kaggle/working/META-CXR-SMOKETEST')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/minhphuong150505/META-CXR-SMOKETEST.git', str(repo_dir)], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'fetch', '--depth=1', 'origin', REPO_COMMIT], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'checkout', '--detach', REPO_COMMIT], check=True)
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
os.environ['PYTHONPATH'] = str(repo_dir) + os.pathsep + os.environ.get('PYTHONPATH', '')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-r', 'requirements-kaggle.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', 'nltk', 'rouge-score', 'pycocoevalcap', 'bert-score'], check=True)
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)


In [ ]:
# Keep credential material outside /kaggle/working, which Kaggle publishes as output.
from smoke.runtime import load_kaggle_secrets
load_kaggle_secrets(('GCS_SERVICE_ACCOUNT', 'WANDB_API_KEY'), '/tmp/.meta-cxr-secrets')
print('Loaded required secrets into OS environment (values hidden).')


In [ ]:
import hashlib, json, torch
from smoke.runtime import discover_dataset, load_dataset_manifest, write_runtime_env_config
from smoke.checkpoints import download_best_checkpoint
dataset_root = discover_dataset(DATASET_SLUG)
dataset_manifest, _, dataset_hash = load_dataset_manifest(dataset_root)
if dataset_manifest.get('status') != 'qa_passed':
    raise RuntimeError('Dataset manifest is not QA-passed')
write_runtime_env_config(dataset_root, '/kaggle/working/meta-cxr-nlp-results')
checkpoint_dir = pathlib.Path('/kaggle/working/meta-cxr-nlp-checkpoint')
checkpoint = download_best_checkpoint(CHECKPOINT_GCS_BUCKET, checkpoint_dir)
checkpoint_meta = torch.load(checkpoint, map_location='cpu', weights_only=False)
identity = checkpoint_meta.get('identity')
if not identity or identity.get('dataset_manifest_sha256') != dataset_hash:
    raise RuntimeError('Checkpoint dataset identity mismatch')
print({'checkpoint_epoch': checkpoint_meta.get('epoch'), 'checkpoint_source_commit': checkpoint_meta.get('source_commit'), 'test_studies': dataset_manifest['counts']['split_studies']['test']})


In [ ]:
from IPython.display import Markdown, display
result_dir = pathlib.Path('/kaggle/working/meta-cxr-nlp-results')
result_path = result_dir / 'report_generation_metrics.json'
env = os.environ.copy()
env['PYTHONPATH'] = str(repo_dir) + os.pathsep + env.get('PYTHONPATH', '')
subprocess.run([sys.executable, 'scripts/evaluate_report_generation.py', '--cfg-path', 'pretraining/configs/stage1_smoke_2xt4.yaml', '--checkpoint', str(checkpoint), '--output', str(result_path), '--overwrite', '--dataset-manifest-sha256', dataset_hash, '--config-fingerprint', identity['config_fingerprint'], '--batch-size', str(BATCH_SIZE), '--num-workers', str(NUM_WORKERS), '--num-beams', str(NUM_BEAMS), '--max-length', str(MAX_LENGTH), '--min-length', str(MIN_LENGTH)], check=True, env=env)
summary = json.loads(result_path.read_text())
metrics = summary['metrics']
lines = ['### Report generation — MIMIC-CXR test set', '', '| BLEU-1 | BLEU-4 | ROUGE-L | METEOR | CIDEr | BERTScore |', '|---:|---:|---:|---:|---:|---:|', f"| {metrics['bleu_1']:.3f} | {metrics['bleu_4']:.3f} | {metrics['rouge_l']:.3f} | {metrics['meteor']:.3f} | {metrics['cider']:.3f} | {metrics['bertscore_f1']:.3f} |"]
display(Markdown('\n'.join(lines)))
print({key: summary[key] for key in ('status', 'test_studies', 'evaluated_reports', 'wall_seconds', 'bertscore_model')})
